<a href="https://colab.research.google.com/github/sahanabalajee/MetaMuseum/blob/sofia/Emotion_related_artwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

New Model


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
image_dir = "/content/drive/My Drive/art_gallery_dataset/original"

In [ ]:
# We are using CLIP because it helps match emotions with images by
# understanding both text and pictures in the same way.
!pip install git+https://github.com/openai/CLIP.git


  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ur9wl5tz
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ur9wl5tz
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# We are using FAISS because it quickly finds the images that best match the emotion.
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 57.1 MB/s eta 0:00:00


In [ ]:
import os
import torch
import clip
from PIL import Image
import faiss
import numpy as np

In [ ]:
# Device configuration :checks if a GPU (CUDA) is available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the CLIP model and preprocessing function : Vision Transformer - 32 * 32 pixel images
model, preprocess = clip.load("ViT-B/32", device=device)

# Build a list of image file paths from the mounted drive directory
image_paths = [
    os.path.join(image_dir, fname)
    for fname in os.listdir(image_dir)
    if fname.lower().endswith(('.png', '.jpg', '.jpeg'))
]

# Store image embeddings and corresponding file paths
image_embeddings = []
indexed_paths = []

# Process each image: load, preprocess, and compute its embedding
for path in image_paths:
    try:
        image = Image.open(path).convert("RGB")
        image_input = preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = model.encode_image(image_input)
        # Normalize the embedding to unit length
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
        image_embeddings.append(embedding.cpu().numpy())
        indexed_paths.append(path)
    except Exception as e:
        print(f"Error processing {path}: {e}")

# Convert list of embeddings to a numpy array of shape (N, embedding_dim)
embeddings_array = np.concatenate(image_embeddings, axis=0)

# Build the FAISS index (using cosine similarity which works with normalized vectors)
d = embeddings_array.shape[1]  # Dimension of embeddings
index = faiss.IndexFlatIP(d)  # Inner product on normalized vectors equals cosine similarity
index.add(embeddings_array)
print(f"FAISS index built with {index.ntotal} images.")

def get_similar_images(emotion_text, top_k=5):
    """
    Given an emotion (as text), compute its embedding using CLIP's text encoder,
    then query the FAISS index for the top_k similar images.
    """
    # Tokenize and encode the emotion text
    text_input = clip.tokenize([emotion_text]).to(device)
    with torch.no_grad():
        text_embedding = model.encode_text(text_input)
    # Normalize the text embedding
    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
    text_embedding_np = text_embedding.cpu().numpy()

    # Search in the FAISS index (using inner product for normalized vectors)
    distances, indices = index.search(text_embedding_np, top_k)
    similar_images = [indexed_paths[idx] for idx in indices[0]]
    return similar_images, distances[0]

# --- Example Usage ---
if __name__ == "__main__":
    emotion = input("Enter an emotion (e.g., 'sadness', 'joy'): ")
    images, scores = get_similar_images(emotion)
    print("Top 5 images for emotion '{}':".format(emotion))
    for i, (img_path, score) in enumerate(zip(images, scores)):
        print(f"{i+1}. {img_path} (similarity score: {score:.4f})")


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 170MiB/s]


FAISS index built with 400 images.


In [ ]:
import matplotlib.pyplot as plt

def display_images(image_paths):
    """
    Displays the retrieved images in a single row.
    """
    fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 5))

    for ax, img_path in zip(axes, image_paths):
        image = Image.open(img_path)
        ax.imshow(image)
        ax.axis("off")

    plt.show()

# Example usage
emotion = input("Enter an emotion (e.g., 'sadness', 'joy'): ")
images, scores = get_similar_images(emotion)

print("Top 5 images for emotion '{}':".format(emotion))
display_images(images)
